# Chatbot de Inteligencia Financiera

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Construir un modelo semántico** en YAML para Cortex Analyst
2. **Habilitar consultas en lenguaje natural** sobre datos financieros
3. **Crear interfaz conversacional** con `CORTEX.COMPLETE()`
4. **Definir métricas de negocio** en la capa semántica
5. **Desplegar analytics self-service** para usuarios no técnicos

---

## Lo Que Construirás

Un **chatbot financiero conversacional** que responde preguntas de negocio:
- Traducción de lenguaje natural a SQL
- Modelo semántico con métricas bancarias
- Respuestas contextualizadas con datos reales
- Interfaz de chat interactiva

**Valor de Negocio**: Democratizar el acceso a datos financieros sin necesidad de SQL.

---

## Funcionalidades Clave

- Cortex Analyst — Motor NL-to-SQL
- Modelos Semánticos (YAML) — Definiciones de negocio
- `CORTEX.COMPLETE()` — Respuestas en lenguaje natural
- Streamlit — Interfaz de chat interactiva

¡Comencemos!

---

## Paso 1: Configuración del Entorno

### Cortex Analyst para Banca

**Objetivo**: Permitir a gestores y directivos consultar datos financieros en lenguaje natural.

In [ ]:
CREATE DATABASE IF NOT EXISTS CHATBOT_FINANCIERO_DB;
CREATE SCHEMA IF NOT EXISTS CHATBOT_FINANCIERO_DB.PUBLIC;
USE SCHEMA CHATBOT_FINANCIERO_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() AS db;

---

## Paso 2: Crear Tablas de Datos Financieros

### Tablas

1. **`OPERACIONES_DIARIAS`** — Transacciones bancarias diarias
2. **`PRODUCTOS_BANCARIOS`** — Catálogo de productos
3. **`OBJETIVOS_COMERCIALES`** — Targets por producto y oficina

In [ ]:
CREATE OR REPLACE TABLE OPERACIONES_DIARIAS (
    fecha DATE,
    producto VARCHAR(50),
    oficina VARCHAR(50),
    gestor VARCHAR(100),
    num_operaciones INTEGER,
    volumen_eur DECIMAL(15,2),
    comisiones_eur DECIMAL(10,2)
);

CREATE OR REPLACE TABLE OBJETIVOS_COMERCIALES (
    mes DATE,
    producto VARCHAR(50),
    oficina VARCHAR(50),
    objetivo_volumen DECIMAL(15,2),
    objetivo_comisiones DECIMAL(10,2)
);

---

## Paso 3: Generar Datos Financieros

### Qué Vamos a Crear

- **12 meses** de operaciones diarias
- **5 productos** bancarios (Hipotecas, Préstamos, Fondos, Seguros, Tarjetas)
- **8 oficinas** principales

In [ ]:
-- Generar operaciones diarias (12 meses)
INSERT INTO OPERACIONES_DIARIAS
WITH fechas AS (
    SELECT DATEADD('day', -SEQ4(), CURRENT_DATE()) AS fecha
    FROM TABLE(GENERATOR(ROWCOUNT => 365))
),
productos AS (SELECT column1 AS producto FROM VALUES ('Hipotecas'),('Préstamos'),('Fondos Inversión'),('Seguros'),('Tarjetas')),
oficinas AS (SELECT column1 AS oficina FROM VALUES ('Madrid Centro'),('Barcelona Diagonal'),('Valencia Plaza'),('Sevilla Nervión'),('Bilbao Gran Vía'),('Zaragoza Independencia'),('Málaga Larios'),('Palma Jaime III'))
SELECT
    f.fecha, p.producto, o.oficina,
    'Gestor ' || LPAD(UNIFORM(1,20,RANDOM())::STRING, 2, '0'),
    UNIFORM(5, 50, RANDOM()),
    ROUND(UNIFORM(50000, 2000000, RANDOM()), 2),
    ROUND(UNIFORM(500, 15000, RANDOM()), 2)
FROM fechas f CROSS JOIN productos p CROSS JOIN oficinas o
WHERE UNIFORM(0,1,RANDOM()) > 0.92;

-- Generar objetivos
INSERT INTO OBJETIVOS_COMERCIALES
WITH meses AS (SELECT DATE_TRUNC('month', DATEADD('month', -SEQ4(), CURRENT_DATE())) AS mes FROM TABLE(GENERATOR(ROWCOUNT => 12)))
SELECT m.mes, p.producto, o.oficina,
    ROUND(UNIFORM(5000000, 20000000, RANDOM()), 2),
    ROUND(UNIFORM(50000, 200000, RANDOM()), 2)
FROM meses m
CROSS JOIN (SELECT column1 AS producto FROM VALUES ('Hipotecas'),('Préstamos'),('Fondos Inversión'),('Seguros'),('Tarjetas')) p
CROSS JOIN (SELECT column1 AS oficina FROM VALUES ('Madrid Centro'),('Barcelona Diagonal'),('Valencia Plaza'),('Sevilla Nervión'),('Bilbao Gran Vía'),('Zaragoza Independencia'),('Málaga Larios'),('Palma Jaime III')) o;

SELECT COUNT(*) AS operaciones FROM OPERACIONES_DIARIAS;

---

## Paso 4: Crear Modelo Semántico

### Qué Vamos a Crear

Un **modelo semántico** en YAML que mapea terminología de negocio a esquemas de base de datos.

### ¿Por Qué Modelos Semánticos?

- Mapea "ingresos" → `SUM(comisiones_eur)`
- Mapea "mes pasado" → `WHERE fecha >= DATE_TRUNC('month', DATEADD('month', -1, CURRENT_DATE()))`
- Define dimensiones, métricas y relaciones válidas

**Impacto**: Usuarios no técnicos consultan datos sin escribir SQL.

In [ ]:
-- Crear stage para modelo semántico
CREATE SCHEMA IF NOT EXISTS CHATBOT_FINANCIERO_DB.SEMANTIC_MODELS;
USE SCHEMA CHATBOT_FINANCIERO_DB.SEMANTIC_MODELS;

CREATE OR REPLACE STAGE model_stage
    DIRECTORY = (ENABLE = TRUE)
    COMMENT = 'Modelos semánticos para Cortex Analyst';

-- Mostrar contenido del modelo semántico YAML
SELECT
'name: banking_analytics\n' ||
'description: Datos operativos y comerciales bancarios\n' ||
'tables:\n' ||
'  - name: operaciones_diarias\n' ||
'    base_table:\n' ||
'      database: CHATBOT_FINANCIERO_DB\n' ||
'      schema: PUBLIC\n' ||
'      table: OPERACIONES_DIARIAS\n' ||
'    dimensions:\n' ||
'      - name: fecha\n' ||
'        data_type: DATE\n' ||
'      - name: producto\n' ||
'        synonyms: [tipo de producto, línea de negocio]\n' ||
'        data_type: VARCHAR\n' ||
'      - name: oficina\n' ||
'        synonyms: [sucursal, branch]\n' ||
'        data_type: VARCHAR\n' ||
'      - name: gestor\n' ||
'        synonyms: [representante, asesor]\n' ||
'        data_type: VARCHAR\n' ||
'    measures:\n' ||
'      - name: volumen_eur\n' ||
'        synonyms: [volumen, importe, revenue]\n' ||
'        aggregation: sum\n' ||
'      - name: comisiones_eur\n' ||
'        synonyms: [comisiones, ingresos, fees]\n' ||
'        aggregation: sum\n' ||
'      - name: num_operaciones\n' ||
'        synonyms: [operaciones, transacciones]\n' ||
'        aggregation: sum' AS yaml_content;

---

## Paso 5: Consultas con Lenguaje Natural

### `CORTEX.COMPLETE()` para Inteligencia Financiera

Usamos el LLM para responder preguntas de negocio con contexto de datos reales.

In [ ]:
-- Preguntas de ejemplo con respuesta IA
CREATE OR REPLACE TABLE PREGUNTAS_EJEMPLO (
    pregunta_id INTEGER,
    pregunta TEXT
);

INSERT INTO PREGUNTAS_EJEMPLO VALUES
(1, '¿Cuáles fueron las comisiones totales del último mes por producto?'),
(2, '¿Qué oficina tiene mayor volumen de hipotecas?'),
(3, '¿Cuál es el gestor con más operaciones de fondos?'),
(4, 'Compara el rendimiento de Madrid Centro vs Barcelona Diagonal'),
(5, '¿Estamos cumpliendo los objetivos de tarjetas?');

-- Generar respuestas con datos reales
CREATE OR REPLACE TABLE RESPUESTAS_CHATBOT AS
WITH contexto AS (
    SELECT
        'Datos bancarios disponibles:\n' ||
        'Productos: Hipotecas, Préstamos, Fondos Inversión, Seguros, Tarjetas\n' ||
        'Oficinas: Madrid Centro, Barcelona Diagonal, Valencia, Sevilla, Bilbao, Zaragoza, Málaga, Palma\n' ||
        'Período: últimos 12 meses\n' ||
        'Volumen total: ' || (SELECT TO_CHAR(SUM(volumen_eur),'999,999,999,999') FROM CHATBOT_FINANCIERO_DB.PUBLIC.OPERACIONES_DIARIAS) || '€\n' ||
        'Comisiones totales: ' || (SELECT TO_CHAR(SUM(comisiones_eur),'999,999,999') FROM CHATBOT_FINANCIERO_DB.PUBLIC.OPERACIONES_DIARIAS) || '€' AS ctx
)
SELECT
    p.pregunta_id, p.pregunta,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large',
        c.ctx || '\n\nPregunta: ' || p.pregunta || '\n\nResponde de forma concisa en español con datos específicos.'
    ) AS respuesta
FROM PREGUNTAS_EJEMPLO p, contexto c;

SELECT * FROM RESPUESTAS_CHATBOT;

---

## Paso 6: Dashboard Chatbot Interactivo

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Chatbot de Inteligencia Financiera')
st.markdown('### Consulta datos bancarios en lenguaje natural')

st.subheader('Preguntas de Ejemplo')
ejemplos = session.sql('SELECT pregunta FROM PREGUNTAS_EJEMPLO').to_pandas()
for _, row in ejemplos.iterrows():
    if st.button(row['PREGUNTA'][:80]):
        st.session_state['q'] = row['PREGUNTA']

st.subheader('Tu Pregunta')
user_q = st.text_input('Escribe tu pregunta', value=st.session_state.get('q',''))

if user_q:
    ctx = session.sql("SELECT 'Volumen total: ' || TO_CHAR(SUM(volumen_eur),'999,999,999,999') || ' EUR, Comisiones: ' || TO_CHAR(SUM(comisiones_eur),'999,999,999') || ' EUR' FROM OPERACIONES_DIARIAS").collect()[0][0]
    resp = session.sql(f"SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large', '{ctx}\\n\\nPregunta: {user_q}\\n\\nResponde en español.')").collect()[0][0]
    st.markdown('**Respuesta:**')
    st.write(resp)

st.markdown('---')
st.subheader('KPIs Generales')
df = session.sql('SELECT producto, SUM(volumen_eur) AS volumen, SUM(comisiones_eur) AS comisiones FROM OPERACIONES_DIARIAS GROUP BY 1 ORDER BY volumen DESC').to_pandas()
st.dataframe(df)

---

## Paso 7: Limpieza (Opcional)

In [ ]:
-- Descomentar para limpiar

-- DROP DATABASE IF EXISTS CHATBOT_FINANCIERO_DB;